# L10a: Online Learning: Weighted Majority and Multiplicative Weights Algorithms
In this lecture, we introduce online learning from expert advice: we combine the predictions of several experts and update each expert's influence from its past performance.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Describe the online learning problem:__ [Online learning](https://en.wikipedia.org/wiki/Online_machine_learning) is a type of machine learning where the model learns from data as it arrives, rather than from a fixed dataset. We'll set up online learning as a game played over many rounds, in which we make a prediction, an adversary reveals the outcome, and we learn from the result.
> * __Measure an online learning algorithm with regret:__ [Regret](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2025/blob/main/lectures/week-6/L6a/docs/Orabona-OnlineLearning-AnnRev-2021.pdf) is the difference between the loss of an algorithm and the loss of the best fixed strategy in hindsight. We'll define the regret of an online learning algorithm and see why we want a bound on regret that grows more slowly than the number of rounds.
> * __Learn from expert advice with multiplicative weights:__ The [weighted majority and multiplicative weights algorithms](https://en.wikipedia.org/wiki/Multiplicative_weight_update_method) keep a weight for each expert and shrink the weight of an expert each time it does poorly. We'll work through both algorithms and the bounds that compare their performance with that of the best expert.

Online learning is a foundational component of reinforcement learning, and it is such an intuitively simple idea: a machine (like us from when we are a baby to adulthood) learns from experience, good and bad. The idea dates [back to the 1950s and has been studied extensively in the context of game theory, optimization, and machine learning](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2025/blob/main/lectures/week-6/L6a/docs/Orabona-OnlineLearning-arXiv-2019.pdf).

I'm super excited to share this module with you, so let's get started!
___

## General Reinforcement/Online Learning Problem
Suppose we have an agent that can be in a state $s \in \mathcal{S}$, where $\mathcal{S}$ is the set of possible states, and can take an action $a \in \mathcal{A}$, where $\mathcal{A}$ is the set of possible actions. After taking action $a$ in state $s$, the agent receives a reward $r$ and moves to a new state $s'$. How does the agent learn which action to take in each state to maximize its cumulative reward?

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-Schematic-RL.svg" width="580" style="max-width:100%; height:auto;" alt="Loop diagram. A blue Agent box at the top sends an arrow labeled action a down the right side to a green Environment box at the bottom. Two arrows return up the left side from the Environment to the Agent, labeled reward r and state s to s prime."/>
</p>

Three families of methods address this problem:

* __Multiplicative weights__ update expert weights based on their past performance.
* __Bandit algorithms__ observe only the reward of the action they take, and in the basic multi-armed bandit there is no evolving state. They try different actions to estimate their rewards and favor the actions that pay off.
* __Q-learning__ is a value-based method that estimates the long-term value of each state-action pair, so the agent can learn optimal behavior when its actions change the states it visits next.


The feedback available to the agent determines how it can learn. With expert advice, the revealed outcome lets us score every expert after each round. With bandit feedback, we see only the reward of the action we chose, so each choice is also an experiment. In Q-learning, each action also changes the next state. We begin with expert advice, the full-feedback case.

___

## Online Learning and Regret Minimization
Suppose we play a number-guessing game over $T$ rounds.

For $t = 1, 2, ..., T$ __do__:
1. An _adversary_ picks a number $y_t$ from the set $\{1, 2, ..., N\}$ and keeps it secret.
2. The _aggregator_ (us) makes a prediction $\hat{y}_t$ for the number, based on some strategy or algorithm.
3. The adversary reveals the true number $y_t$, and we receive feedback using a loss function $l_t(\hat{y}_t, y_{t})$, which measures the discrepancy between our prediction and the true number.

The gap between our cumulative loss and that of the __best fixed strategy in hindsight__ is the _regret_:
$$
\begin{equation*}
R(T) = \underbrace{\sum_{t=1}^{T} l_t(\hat{y}_t, y_t)}_{\text{aggregator}} - \underbrace{\min_{u} \sum_{t=1}^{T} l_t(u,y_t)}_{\text{best fixed strategy in hindsight}}
\end{equation*}
$$
where the minimum is taken over every fixed prediction $u$ that we could have made in all $T$ rounds. Online learning seeks algorithms that _minimize cumulative regret_, so they perform _nearly_ as well as the best fixed strategy in hindsight.

> __Key distinctions__
> 
> * __Adaptation__: Online algorithms update their predictions one round at a time, and their guarantees do not assume the data come from a fixed distribution. Batch learning algorithms, such as ordinary least squares, are trained once on a fixed dataset.
> * __Regret minimization__: Regret depends on the _algorithm_ and on the sequence of outcomes. A regret bound holds for every sequence, so it is a property of the algorithm, and we want algorithms with small regret bounds.

With expert advice, the best fixed strategy is the best single expert: the expert with the smallest total loss, even though its predictions change from round to round.

___

## Weighted Majority Algorithm (WMA)
Let's start with the __Weighted Majority Algorithm (WMA)__ developed by Littlestone and Warmuth in 1994:
* [Littlestone, N., & Warmuth, M. K. (1994). The weighted majority algorithm. Information and Computation, 108(2), 212-261.](https://doi.org/10.1006/inco.1994.1009)

We illustrate it with a _prediction from expert advice_ problem from [Arora et al., 2005, Princeton](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2025/blob/main/lectures/week-6/L6a/docs/Arora-MWsurvey-CS-Princeton.pdf):

* __Game__: Each morning, we predict whether a stock price will go `{up | down}`, a _binary_ outcome we encode as `{1 | -1}`, and at the market close we see the `true` movement. _If our prediction is wrong, we lose a dollar_, so we want to _minimize_ our losses. We get advice from $n$ experts, whose predictions may be correlated and may or may not be correct.
* __Goal__: Keep our mistakes within a multiple of the _best_ expert's mistakes plus a constant, without knowing which expert is best until the game ends. We keep a weight for each expert: all start equal, and the experts who predict better gain a larger share of the total weight.

The game runs over $T$ rounds $t = 1, 2, \ldots, T$ between an omniscient _adversary_ (the market) and an _aggregator_ (us). 

In each round, we make a _binary_ prediction $\hat{y}_t \in \{-1, 1\}$, and the adversary reveals the true outcome $y_t$.  A prediction, ours or an expert's, is a _mistake_ when it differs from $y_t$. The experts start with weights $\left\{w_{i}^{(1)} = 1 \mid i = 1, 2, \ldots, n\right\}$, and we predict `-1` when the weighted vote is tied.


### Algorithm

__Initialize__: Given $n$ experts, the number of rounds $T\geq1$, and a learning rate $0<\epsilon\leq{1}/{2}$, set the weight $w_{i}^{(1)}\gets1$ for each expert $i=1,2,\ldots,n$.

For each round $t=1,2,\dots,T$ __do__:
1. The aggregator (us) makes a prediction $\hat{y}_t \in \{-1, 1\}$ based on the weighted majority of the experts' predictions:
   - If the total weight of the experts predicting `1` at time $t$ is greater than the total weight of the experts predicting `-1` __then__ predict $\hat{y}_t\gets1$.
   - Otherwise, predict $\hat{y}_t\gets-1$.
2. The adversary (market) reveals the actual outcome $y_t \in \{-1, 1\}$.
3. We decrease the weights of the experts who predicted incorrectly. For each expert $i=1,2,\ldots,n$:
   - If expert $i$ predicted incorrectly __then__ set $w_{i}^{(t+1)}\gets w_{i}^{(t)}(1-\epsilon)$.
   - Otherwise, keep its weight, $w_{i}^{(t+1)}\gets w_{i}^{(t)}$.


### Worked Example: Four Rounds with Three Experts
Three experts predict a binary outcome over four rounds, with $\epsilon=1/2$, so each mistake halves an expert's weight. The table lists the weights **before** each round, the expert predictions, our weighted vote, and the outcome:

| Round | Weights before the round | Expert predictions | Our prediction | Actual outcome |
|---|---|---|---:|---:|
| 1 | (1, 1, 1) | (1, 1, −1) | 1 | 1 |
| 2 | (1, 1, 0.5) | (−1, 1, 1) | 1 | −1 |
| 3 | (1, 0.5, 0.25) | (−1, 1, 1) | −1 | 1 |
| 4 | (0.5, 0.5, 0.25) | (1, −1, 1) | 1 | 1 |

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-WMA-Weights/Fig-WMA-Weights.svg" width="620" style="max-width:100%; height:auto;" alt="Weighted vote in each of the four rounds of the worked example, drawn as a tug of war across a center line, down to the left and up to the right. Each expert is a numbered block, as wide as its weight, on the side it predicted, and the outcome's side is shaded. Columns to the right, headed weight voting, list the summed weight of the experts voting down and voting up, with the heavier total in bold, then our prediction and the outcome. Round 1: experts 1 and 2 up, total 2, against expert 3 down, total 1; we predict up, the outcome is up, and expert 3 is outlined in red as a mistake. Round 2: experts 2 and 3 up, total 1.5, against expert 1 down, total 1; we predict up, the outcome is down, and experts 2 and 3 and our prediction are outlined in red. Round 3: experts 2 and 3 up, now total 0.75, against expert 1 down, total 1; we predict down, the outcome is up, and expert 1 and our prediction are outlined in red. Round 4: experts 1 and 3 up, total 0.75, against expert 2 down, total 0.5; we predict up, the outcome is up, and expert 2 is outlined in red. A key explains the shading, the red outline for a mistake whose weight halves, and a scale bar for weight 1."/>
</p>

__The weighted vote in the worked example.__ Each block is an expert, as wide as its weight, on the side it predicted, and the heavier side is our prediction. The weight voting columns add up the weights of the experts on each side. Blocks on the side opposite the outcome are mistakes, outlined in red, and they are half as wide in the next round.

After round 4, the weights are $(0.5,0.25,0.25)$: expert 1 made one mistake, and experts 2 and 3 each made two. We made two, in rounds 2 and 3. With a loss of one per mistake, our regret against the best fixed expert is given by:
$$
R_{\mathrm{WMA}}(4)=2-\min\{1,2,2\}=1.
$$
By the end, the weights point to the best expert, but our early mistakes still count. The mistake guarantee bounds that loss for any sequence of predictions and outcomes.


### Mistake Guarantee

__Theorem__: Let $M_{i}^{(t)}$ be the number of mistakes expert $i$ makes up to time $t$, and $M^{(t)}$ the number of mistakes the aggregator (us) makes. Then, for every expert $i$, the weighted majority algorithm guarantees:
$$
\begin{align*}
M^{(t)} \leq \frac{2\ln(n)}{\epsilon} + 2\left(1+\epsilon\right)M_{i}^{(t)}.
\end{align*}
$$

[Arora et al. prove this bound](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2025/blob/main/lectures/week-6/L6a/docs/Arora-MWsurvey-CS-Princeton.pdf) by tracking how the total weight of the experts shrinks. Two features stand out:

* __Multiplicative:__ our mistakes are at most $2(1+\epsilon)$ times the best expert's, plus the constant $2\ln(n)/\epsilon$, which depends on the number of experts and the learning rate but not on the number of rounds.
* __Deterministic:__ the same predictions and outcomes always give the same votes.

__Why choose an expert at random?__ The adversary knows our rule, so it can predict our vote and pick the opposite outcome, making us wrong every round. With two experts who always disagree, the better one is wrong at most half the time, so no deterministic rule can avoid making twice the best expert's mistakes.

Choosing an expert at random takes away this advantage.

___

## Multiplicative Weights Algorithm (MWA)
The **Multiplicative Weights Algorithm (MWA)** also learns from expert advice, but instead of taking a weighted vote, it follows one expert chosen at random in proportion to its weight. Its learning rate $\eta$ plays the role of $\epsilon$ and controls how quickly the weights adapt. Let's walk through the setup and the algorithm.


### Problem Setting
In each round $t = 1, 2, \ldots, T$, each of $n$ experts makes a recommendation, and we want to combine their advice into good decisions even when an adversary picks the outcomes. We assume the adversary fixes the advice and outcomes in advance, so they do not depend on which expert we draw.

* Let $\mathbf{p}^{(t)} = \{p_1^{(t)}, p_2^{(t)}, \ldots, p_n^{(t)}\}$ denote our distribution over experts at round $t$, built from their past performance.
* We sample an expert from this distribution, $i \sim \texttt{Categorical}(\mathbf{p}^{(t)})$, and follow its recommendation.
* After the adversary reveals the true outcome, we compute the cost vector $\mathbf{m}^{(t)} = \{m_1^{(t)}, \dots, m_n^{(t)}\}$, where $m_i^{(t)} \in [-1, 1]$ is the cost of expert $i$ at time $t$: $-1$ for a correct prediction and $+1$ for an incorrect one.

The **expected cost** of the algorithm at round $t$ is given by:
$$
L^{(t)} = \sum_{i=1}^{n} p_i^{(t)} m_i^{(t)}.
$$
The **total expected cost** over $T$ rounds is given by:
$$
L_A = \sum_{t=1}^{T} L^{(t)} = \sum_{t=1}^{T} \sum_{i=1}^{n} p_i^{(t)} m_i^{(t)}.
$$
We want this total to be as close as possible to that of the **best expert in hindsight**, and we call the difference the **regret**:
$$
R(T) = \underbrace{\sum_{t=1}^{T} \sum_{i=1}^{n} p_i^{(t)} m_i^{(t)}}_{\text{Algorithm's Loss}} - \underbrace{\min_{i \in \{1, \dots, n\}} \sum_{t=1}^{T} m_i^{(t)}}_{\text{Best Expert's Loss}}.
$$
Good online learning algorithms such as MWA have a **sublinear** regret bound, one that grows more slowly than the number of rounds $T$, so the bound on the average regret per round, $R(T)/T$, vanishes as $T\to\infty$.


### Algorithm

__Initialize__: Given $n$ experts, the number of rounds $T\geq1$, and a learning rate $0<\eta\leq{1}/{2}$, set the weight $w_{i}^{(1)}\gets1$ for each expert $i=1,2,\ldots,n$.

For $t=1,2,\dots,T$ __do__:
1. Ask each expert $i$ what the outcome of the experiment should be, and denote the expert's answer as $\hat{y}_{i}^{(t)}$. Choose expert $i$ with probability $p_{i}^{(t)} = w_{i}^{(t)}/\sum_{j=1}^{n}w_{j}^{(t)}$ and follow its answer.
2. The adversary (nature) reveals the true outcome $y_{t}$ of the experiment at time $t$. Compute the cost $m_{i}^{(t)}$ of each expert $i$, using that expert's answer $\hat{y}_{i}^{(t)}$:
    $$
    m_i^{(t)} =
    \begin{cases}
    -1 & \text{if } \hat{y}_i^{(t)} = y_t \quad \text{(correct)} \\
    +1 & \text{if } \hat{y}_i^{(t)} \neq y_t \quad \text{(incorrect)}
    \end{cases}.
   $$
3. Update the weight of each expert $i$ (we normalize the weights when we form the next probability distribution):
   $$
   w_{i}^{(t+1)}\gets w_{i}^{(t)}\cdot\left(1-\eta\cdot{m_{i}^{(t)}}\right).
   $$

This is a super simple algorithm with some very nice properties. Each weight is multiplied by a factor set by that expert's cost, hence the name, and $\eta$ controls how quickly the weights shift toward the better experts. The algorithm also comes with a guarantee that it does nearly as well as the best fixed expert in hindsight; let's check that out!


### Theoretical Regret Bound
Assume all costs lie in $m_i^{(t)} \in [-1, 1]$ and fix a learning rate $0<\eta \leq \frac{1}{2}$. For any expert $i$ after $T$ rounds, MWA guarantees the first inequality. Taking $i$ to be the best expert in hindsight and moving its cost to the left side gives the regret:
$$
\begin{align*}
\underbrace{\sum_{t=1}^{T} \mathbf{p}^{(t)} \cdot \mathbf{m}^{(t)}}_{\text{our expected cost}} & \leq \underbrace{\sum_{t=1}^{T} m_i^{(t)}}_{\text{expert } i\text{'s cost}} + \underbrace{\eta \sum_{t=1}^{T} |m_i^{(t)}|}_{\text{learning too fast}} + \underbrace{\frac{\ln n}{\eta}}_{\text{learning too slowly}} \\
\underbrace{\sum_{t=1}^{T} \mathbf{p}^{(t)} \cdot \mathbf{m}^{(t)} - \overbrace{\sum_{t=1}^{T} m_i^{(t)}}^{\text{best expert}}}_{R(T)} & \leq \eta T + \frac{\ln n}{\eta} \\
R(T) & \leq \eta T + \frac{\ln n}{\eta},
\end{align*}
$$
where we used $|m_i^{(t)}|\leq1$; equality holds for our binary signed costs. A large $\eta$ overreacts to each round's costs, while a small $\eta$ moves weight away from the equal starting weights slowly, and the choice of $\eta$ balances these two terms. By choosing $\eta = \sqrt{\frac{\ln n}{T}}$, which satisfies $\eta\leq{1/2}$ when $T\geq 4\ln n$, this regret bound becomes sublinear:
$$
R(T) \leq 2 \sqrt{T \ln n}.
$$
Thus, the average regret per round is at most $2\sqrt{\ln n/T}$, which vanishes as $T\to\infty$, so MWA performs nearly as well as the best fixed expert in hindsight.


### Compare the Two Guarantees
Both algorithms use **full feedback**: once the outcome is revealed, we score and update every expert. They differ in how they choose a prediction and in the performance quantity they bound:

| Algorithm | Prediction rule | Performance guarantee |
|---|---|---|
| Weighted majority | Take the weighted binary vote | Our mistakes are at most $2(1+\epsilon)$ times the best expert's, plus $2\ln n/\epsilon$. |
| Multiplicative weights | Sample an expert in proportion to its weight | Our expected cost exceeds the best expert's by at most $2\sqrt{T\ln n}$, with $\eta=\sqrt{\ln n/T}$ and $T\geq 4\ln n$. |

__How do the two loss conventions compare?__ WMA counts mistakes, while MWA uses signed costs: $+1$ for a mistake and $-1$ for a correct prediction. A record of $M$ mistakes in $T$ rounds has signed total $2M - T$, so the $T$ cancels in the regret and the signed-cost regret is twice the mistake regret. In mistakes, the MWA bound is $\sqrt{T\ln n}$. [Arora et al., Sections 1–2](https://theoryofcomputing.org/articles/v008a006/v008a006.pdf) develop both algorithms in more detail.


### Additional Resources
This module borrowed notes and was inspired from several sources: [Arora et al., The Multiplicative Weights Update Method: A Meta-Algorithm and Applications, Theory of Computing, Volume 8 (2012), pp. 121–164](https://theoryofcomputing.org/articles/v008a006/v008a006.pdf) and the [15-859 CMU Lecture 16](https://www.cs.cmu.edu/afs/cs.cmu.edu/academic/class/15859-f11/www/notes/lecture16.pdf) and [15-859 CMU Lecture 17](https://www.cs.cmu.edu/afs/cs.cmu.edu/academic/class/15859-f11/www/notes/lecture17.pdf).

___

## Lab
In the [lab](../L10b/CHEME-5800-L10b-Lab-WeightedMajorityAlgorithm-Fall-2026.ipynb), we implement the weighted majority algorithm and use it to combine the advice of several experts, each a ticker such as QQQ or AAPL, on whether the SPY exchange-traded fund goes up or down each trading day. We then check the mistake bound, first when the experts see the same day's price moves and then when they must forecast the next day.

___

## Summary
In this lecture, we introduced online learning and two algorithms that learn from expert advice.

> __Key Takeaways:__
>
> * __Online learning and regret:__ We set up online learning as a repeated game in which an aggregator makes a prediction, an adversary reveals the outcome, and the aggregator learns from the result. We measured the performance of an algorithm with its regret, the gap between its total loss and that of the best fixed strategy in hindsight, and looked for a regret bound that grows more slowly than the number of rounds.
>
> * __The weighted majority algorithm:__ We kept a weight for each expert, predicted with the weighted majority vote, and reduced the weight of every expert that made a mistake. The theorem bounds the number of mistakes we make by a term that grows with the logarithm of the number of experts plus two to three times the mistakes of the best expert, even though we do not know in advance which expert is best.
>
> * __The multiplicative weights algorithm:__ We turned the expert weights into a probability distribution, followed an expert chosen at random from it, and updated every expert's weight by its cost. With a learning rate chosen for the number of rounds, the regret bound grows only with the square root of the number of rounds, so the algorithm does nearly as well as the best expert.

Both algorithms rely on full feedback: once the outcome is revealed, we can score every expert, not only the one we followed.

___